# Sampling experiments

Current code includes the corrections documented in [research notes](../docs/research-notes.md). Historical evidence remains in [the report](../docs/report.pdf) and exported figures; reruns can differ.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "src/samplers.py").is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from samplers import uniform, exponential, negative_binomial, generate_normal_box_muller, gamma_rejection

import numpy as np
np.random.seed(42)


## Setup and plotting helpers

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binom, nbinom, hypergeom, expon, gamma as gamma_dist, beta as beta_dist, norm
from scipy.special import comb, gamma
from scipy.optimize import minimize_scalar

# Global functions to plot
SEED = 42
np.random.seed(SEED)

plt.style.use('default')
plt.rcParams.update({
    'figure.figsize': (7, 4.5),
    'axes.grid': True,
    'grid.linestyle': ':',
    'axes.titlesize': 'large',
    'axes.labelsize': 'medium',
})

BAR_W = 0.35
ALPHA_THEO = 0.9
ALPHA_EMP = 0.6
COLOR_THEO, COLOR_EMP = plt.cm.Set2.colors[:2]


def plot_bar(k, theo, emp, title, xlabel, xlim=None, ylabel='Probability'):
    plt.bar(k - BAR_W/2, theo, BAR_W, alpha=ALPHA_THEO, color=COLOR_THEO, label='Theoretical')
    plt.bar(k + BAR_W/2, emp, BAR_W, alpha=ALPHA_EMP, color=COLOR_EMP, label='Empirical')
    if xlim:
        plt.xlim(*xlim)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_hist(samples, x_grid, pdf, title,
              *, bins=30, xmin=None, xmax=None, xlabel='x'):
    plt.hist(samples,
             bins=bins,
             density=True,
             alpha=ALPHA_EMP,
             color=COLOR_EMP,
             edgecolor='black',
             linewidth=1.0,
             label='Empirical')
    plt.plot(x_grid, pdf,
             color=COLOR_THEO,
             lw=2.0,
             label='Theoretical')

    if xmin is not None or xmax is not None:
        cur_min, cur_max = plt.xlim()     
        if xmin is not None:
            cur_min = xmin
        if xmax is not None:
            cur_max = xmax
        plt.xlim(cur_min, cur_max)

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel('Density')
    plt.legend()
    plt.tight_layout()
    plt.show()

## Bernoulli

In [ ]:
def bernoulli_sample(p, n):
    return (np.random.rand(n) < p).astype(int)

p = 0.2
N = 10_000
ber_samples = bernoulli_sample(p, N)
counts = np.bincount(ber_samples, minlength=2) / N
plot_bar(np.array([0, 1]), np.array([1 - p, p]), counts,
         f'Bernoulli(p={p}): theory vs. sample (n={N:,})', 'x')

## Binomial

In [ ]:
n, p, N = 10, 0.51, 10_000
bin_samples = np.random.binomial(n, p, size=N)
k = np.arange(n + 1)
emp = np.bincount(bin_samples, minlength=n + 1) / N
plot_bar(k, binom.pmf(k, n, p), emp,
         f'Binomial(n={n}, p={p})', 'k')

## Geometric (trials until success)

In [ ]:
p, N = 0.3, 10_000
geo_samples = np.random.geometric(p, size=N)
k = np.arange(1, geo_samples.max() + 1)
emp = np.bincount(geo_samples, minlength=k.size + 1)[1:] / N
plot_bar(k, (1 - p) ** (k - 1) * p, emp,
         f'Geometric(p={p})', 'k')

## Negative binomial

In [ ]:
r, p, N = 8, 0.27, 10_000
nb_samples = np.random.negative_binomial(r, p, size=N)
k = np.arange(nb_samples.max() + 1)
emp = np.bincount(nb_samples, minlength=k.size) / N
plot_bar(k, nbinom.pmf(k, r, p), emp,
         f'Negative Binomial(r={r}, p={p})', 'k (failures)', xlim=(-0.5, 15))

## Hypergeometric

In [ ]:
N_pop, K_succ, n_draw, N = 50, 15, 10, 10_000
hg_samples = np.random.hypergeometric(K_succ, N_pop - K_succ, n_draw, size=N)
k = np.arange(0, min(K_succ, n_draw) + 1)
emp = np.bincount(hg_samples, minlength=k.size) / N
plot_bar(k,
         hypergeom.pmf(k, N_pop, K_succ, n_draw),
         emp,
         f'Hypergeometric(N={N_pop}, K={K_succ}, n={n_draw})', 'k')

## Uniform

In [ ]:
a, b = 0.0, 1.0
size = 100_000



samples = uniform(a, b, size)


x = np.linspace(a, b, 200)
pdf = np.ones_like(x) / (b - a)
plt.hist(samples, bins='fd', density=True, alpha=ALPHA_EMP, color=COLOR_EMP, label='Empirical')
plt.plot(x, np.ones_like(x) / (b - a), color=COLOR_THEO, lw=2, label='Theoretical')
plt.title(f'Uniform({a}, {b})')
plt.xlabel('x')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.show()

## Exponential

In [ ]:
lam = 1
size = 1_000_00

sample = exponential(lam,size)

x = np.linspace(0, 15, 500)

theoretical_pdf = lam * np.exp(-lam * x)

plot_hist(sample, x, theoretical_pdf,
          title="Empirical vs. Exponential (λ = 1)",
          bins=40,
          xmin=0, xmax=15)

## Gamma: sum of exponentials

In [ ]:
from math import gamma as gamma

size = 100_000
labda =1
a= 0
b =1

sample_1 = exponential(labda,size)
sample_2 = exponential(labda,size)
sample_3 = exponential(labda,size)
gamma_3 = sample_1+sample_2+sample_3



r = 3
x   = np.linspace(0, gamma_3.max(), 500)
pdf = (labda**r) * x**(r - 1) * np.exp(-labda * x) / gamma(r)


plot_hist(gamma_3, x, pdf, 
          title="Empirical vs. Gamma (r = 3)",
          bins=50,
          xmin=0, 
          xmax=15 )

## Gamma: rejection sampling

In [ ]:
from scipy.stats import gamma as gamma_dist
shape, rate, num_sim = 3.4, 1.0, 10_000
samples, diagnostics = gamma_rejection(shape, rate, num_sim)
print(f"Envelope: {diagnostics['envelope']:.3f}")
print(f"Accepted {num_sim} of {diagnostics['proposals']} proposals")
x = np.linspace(0, 15, 400)
plt.figure(figsize=(8, 4))
plt.hist(samples, bins=50, density=True, alpha=0.6, label="Generated samples")
plt.plot(x, gamma_dist.pdf(x, a=shape, scale=1/rate), label="Theoretical PDF")
plt.xlabel("x")
plt.ylabel("Density")
plt.title("Gamma rejection sampling (shape=3.4, rate=1)")
plt.legend()
plt.tight_layout()
plt.show()

## Beta: ratio of gamma draws

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta as beta_dist


def beta_distribution(alpha_param, beta_param, lam=1.0, size=1):

    x = np.random.gamma(shape=alpha_param, scale=1 / lam, size=size)
    y = np.random.gamma(shape=beta_param,  scale=1 / lam, size=size)
    return x / (x + y)


alpha_ = 3
beta_  = 6
n      = 10_000

samples = beta_distribution(alpha_, beta_, size=n)


x  = np.linspace(0, 1, 400)
beta_dist_pdf = beta_dist.pdf(x, alpha_, beta_)
plot_hist(samples, x,beta_dist_pdf,title= "Empirical vs. Beta (3,6)",
          bins=50,
          xmin=0, 
          xmax=1)

## Normal: Box–Muller

In [ ]:
N = 10000

samples = generate_normal_box_muller(N)



x = np.linspace(-5, 5, 1000)
pdf = (1 / np.sqrt(2 * np.pi)) * np.exp(-x**2 / 2)
plot_hist(samples,x,pdf,
          title= "Empirical vs. Normal (0,1)",
          bins=50,
          xmin=-5, 
          xmax=5)

## Poisson: exponential interarrival times

In [ ]:
from scipy.stats import poisson
lam = 4.0        
T = 1.0          
n_samples = 10000  

counts = np.zeros(n_samples, dtype=int)
for i in range(n_samples):
    t = 0.0
    count = 0
    while True:
        t += np.random.exponential(scale=1/lam)
        if t < T:
            count += 1
        else:
            break
    counts[i] = count


k_vals = np.arange(0, counts.max() + 1)
theoretical_pmf = poisson.pmf(k_vals, mu=lam * T)


empirical_freq = np.bincount(counts, minlength=len(k_vals)) / n_samples

plot_bar(k_vals, theoretical_pmf, empirical_freq,
         title="Poisson from Exp (λ = 4, T = 1)",
         xlabel="k",
         xlim=(0, 10),              # tuple, because plot_bar does  *xlim
         ylabel='Probability')

## Gibbs sampling on a triangle

In [ ]:
N_total = 10_000   
burn    = 2_000   


x = np.zeros(N_total)
y = np.zeros(N_total)

x[0] = 0.5
y[0] = 0.25

for t in range(1, N_total):
    x[t] = np.random.uniform(y[t-1], 1)     
    y[t] = np.random.uniform(0, x[t])      

gibbs_sample = np.column_stack((x[burn:], y[burn:]))

m            = N_total - burn
u1, u2       = np.random.uniform(size=m), np.random.uniform(size=m)
x_true       = np.sqrt(u1)
y_true       = u2 * x_true
true_sample  = np.column_stack((x_true, y_true))

plt.figure()
plt.scatter(true_sample[:, 0], true_sample[:, 1], s=5, alpha=0.5)
plt.title("True triangle distribution")
plt.xlabel("X")
plt.ylabel("Y")
plt.tight_layout()
plt.show()

plt.figure()
plt.scatter(gibbs_sample[:, 0], gibbs_sample[:, 1], s=5, alpha=0.5)
plt.title("Gibbs sample after burn‑in")
plt.xlabel("X")
plt.ylabel("Y")
plt.tight_layout()
plt.show()